# Using autofe: propose features, validate them, loop

The worked example is the UCI [Taiwanese Bankruptcy](https://archive.ics.uci.edu/dataset/572/taiwanese+bankruptcy+prediction) data: 6,819 companies, 95 financial ratios, 3.2% bankrupt. The incumbent model uses the profitability / leverage / growth ratios. The **cash-flow family** is a hand-declared candidate batch, with two planted controls - `cand_dup_roa_c`, a near-copy of an incumbent, and `cand_noise`, pure noise - so the screens have a known answer to find.

| Section | Covers |
| --- | --- |
| [1. Inputs](#1) | the materials a run needs, and checking them |
| [2. Feature discovery](#2) | an LLM proposes candidates; each is screened on a small sample |
| [3. XGBoost validation](#3) | the five stages that decide which candidates earn a place |
| [4. The loop](#4) | keep what passed, propose again from the stronger incumbent set |

In [1]:
import json
import os
import sys
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore", message="IProgress not found")   # tqdm, via shap

# The repo root, wherever this notebook is run from.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)   # config paths are relative to the repo root, as from the CLI

from validation import Pipeline, build_dataset, load_config

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_colwidth", 80)
CONFIG = "configs/bankruptcy.yaml"

<a id='1'></a>
## 1. Inputs

A run reads **materials prepared once** by the dataset's prepare notebook - `data/bankruptcy/prepare.ipynb` here, and `data/cdss_us_sbs/prepare.ipynb` is the template for a private dataset - plus **one config** that says how to use them. Nothing is split or sampled at run time: the three tables are read exactly as written.

| Material | Config key | Read by |
| --- | --- | --- |
| `train.csv` · `valid.csv` · `test.csv` | `data.paths` | every stage |
| `column_descriptions.json` | `discovery.column_descriptions_path` | discovery: what each column means |
| `few_shot.csv` | `discovery.few_shot_path` | discovery: the example rows the proposer sees |
| `screen_train.csv` · `screen_valid.csv` | `discovery.screen_paths` | discovery: the rows its screen fits and scores on |

What the config declares about them:

| Key | Here |
| --- | --- |
| `data.target` | `bankrupt` |
| `data.id_cols` | `row_id` - for the leakage check and the few-shot ids |
| `features.base` | empty, so every other numeric column is an incumbent |
| `features.new` | the cash-flow family and the two planted controls |
| `discovery.task_description` | what the data is, in domain terms - the proposer's only briefing |

In [2]:
cfg = load_config(CONFIG)
materials = {
    **{f"data.paths.{name}": path for name, path in cfg.data.paths.items()},
    "discovery.column_descriptions_path": cfg.discovery.column_descriptions_path,
    "discovery.few_shot_path": cfg.discovery.few_shot_path,
    **{f"discovery.screen_paths.{name}": path for name, path in cfg.discovery.screen_paths.items()},
}
for key, path in materials.items():
    status = "ok" if Path(path).exists() else "MISSING - run data/bankruptcy/prepare.ipynb"
    print(f"{key:<36} {path:<46} {status}")

data.paths.train                     data/bankruptcy/train.csv                      ok
data.paths.valid                     data/bankruptcy/valid.csv                      ok
data.paths.test                      data/bankruptcy/test.csv                       ok
discovery.column_descriptions_path   data/bankruptcy/column_descriptions.json       ok
discovery.few_shot_path              data/bankruptcy/few_shot.csv                   ok


In [3]:
dataset = build_dataset(cfg)
print(dataset.describe())
print("declared candidates:", dataset.new_features)

{'rows_per_split': {'train': 4091, 'valid': 1364, 'test': 1364}, 'n_base_features': 84, 'n_new_features': 13, 'target': 'bankrupt'}
declared candidates: ['cash_flow_rate', 'cash_flow_per_share', 'cash_reinvestment_pct', 'cash_total_assets', 'cash_current_liability', 'cash_turnover_rate', 'cash_flow_to_sales', 'cash_flow_to_total_assets', 'cash_flow_to_liability', 'cfo_to_assets', 'cash_flow_to_equity', 'cand_dup_roa_c', 'cand_noise']


In [4]:
descriptions = json.loads(Path(cfg.discovery.column_descriptions_path).read_text())
few_shot = pd.read_csv(cfg.discovery.few_shot_path)

print(f"{len(descriptions)} column descriptions, e.g.")
for column in dataset.base_features[:3]:
    print(f"  {column:<56} {descriptions.get(column, '(none)')}")
first = few_shot.query("batch == 0")
print(f"\nfew-shot rows: {few_shot['batch'].nunique()} batches of {len(first)}; "
      f"batch 0 holds {int(first[cfg.data.target].sum())} bankrupt companies")

97 column descriptions, e.g.
  roa_c_before_interest_and_depreciation_before_interest   ROA(C) before interest and depreciation before interest
  roa_a_before_interest_and_pct_after_tax                  ROA(A) before interest and % after tax
  roa_b_before_interest_and_depreciation_after_tax         ROA(B) before interest and depreciation after tax

few-shot rows: 10 batches of 32; batch 0 holds 16 bankrupt companies


<a id='2'></a>
## 2. Feature discovery

An LLM reads the task description and every incumbent column - its description and its values in the few-shot rows - and proposes a batch of new columns as pandas code. Each proposal runs in a sandbox and is **screened**: an XGBoost model with and without it, fitted on a sample of train and scored on a sample of valid. The screen is cheap and noisy by design; it gives the proposer feedback each round, while the decision is section 3's, on the full splits.

The knobs, under `discovery:` in the config:

| Key | Here | What it does |
| --- | --- | --- |
| `strategy` | `caafe` | how the LLM is asked: `caafe`, `elfgym`, `ferg`, `featllm` or `promptfe` |
| `batch_size` | 5 | proposals per round |
| `max_rounds` | 2 | rounds; each one sees every earlier proposal and its screen score |
| `screen_data` | `files` | what the screen fits and scores on: `splits` (the whole train and valid splits) or `files` (rows the prepare notebook sampled from them) |
| `screen_paths` | `screen_train.csv`, `screen_valid.csv` | those samples, in the same format as the splits - class-balanced here, since 3% positives would leave almost none |
| `llm.model` | `gpt-4o` | which model proposes |

Scoring on valid means the proposer's feedback comes from valid rows, so valid stops being an independent check on the features it proposes; test still is.

In [5]:
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY is not set: put it in .env at the repo root, or export it")
print("LLM credential found")

LLM credential found


In [6]:
from discovery.stage import run_discovery_stage

cfg_discovery = load_config(CONFIG, {"discovery.enabled": True})
discovered = run_discovery_stage(cfg_discovery, dataset,
                                 output_dir=ROOT / "outputs" / "usage_discovery")
print(json.dumps(discovered.summary(), indent=2))

model.params sets colsample_bytree < 1: base and leave_one_in variants differ in column count, so their gini gain includes an offset unrelated to any feature (a constant column measured -0.0055 on this data). The screen strips it for its own delta; set these to 1.0 to remove it from the verdict too.


{
  "enabled": true,
  "proposed": 10,
  "kept": 5,
  "kept_features": [
    "operating_expense_to_total_assets_ratio",
    "liquidity_comparison_ratio",
    "cash_flow_ratio",
    "financial_risk_index",
    "income_efficiency_ratio"
  ],
  "rounds": 2,
  "stopped_because": "max_rounds reached (2)",
  "screen_base_score": 0.8584444444444441
}


In [7]:
ledger = discovered.to_frame()
ledger[["round", "feature_name", "rationale", "input_columns", "delta", "error"]].round(4)

,round,feature_name,rationale,input_columns,delta,error
0,1,operating_expense_to_total_assets_ratio,This feature captures the proportion of an organization's operating expenses...,"operating_expense_rate, total_asset_growth_rate",0.0031,None
1,1,profit_margin_variance,This metric provides information on the divergence between core business pro...,"operating_gross_margin, pre_tax_net_interest_rate, realized_sales_gross_marg...",NaN,Generated feature 'profit_margin_variance' is redundant: |rho|=0.999 against...
2,1,liquidity_comparison_ratio,Gauging liquidity in terms of potential asset expansion paints a picture of ...,"quick_assets_total_assets, total_asset_growth_rate",0.0076,None
3,1,asset_utilization_efficiency,This offers insights into how effectively a company is leveraging its assets...,"total_asset_turnover, inventory_turnover_rate_times",NaN,Generated feature 'asset_utilization_efficiency' is redundant: |rho|=0.941 a...
4,1,overall_leverage_factor,"This captures the company's exposure to debt in relation to its equity, high...","debt_ratio_pct, net_value_per_share_a",NaN,Generated feature 'overall_leverage_factor' is redundant: |rho|=0.944 agains...
5,2,cash_flow_ratio,This ratio provides insights into the company's ability to cover its short-t...,"quick_assets_total_assets, current_liability_to_current_assets",0.0022,None
6,2,net_profit_margin_analysis,"This captures how net profitability deviates from operating profitability, i...","pre_tax_net_interest_rate, operating_profit_rate",NaN,Generated feature 'net_profit_margin_analysis' is redundant: |rho|=0.929 aga...
7,2,financial_risk_index,Combining leverage and liabilities provides a comprehensive look at financia...,"degree_of_financial_leverage_dfl, liability_to_equity",-0.0009,None
8,2,short_term_funding_capacity,This measures how well the company can cover its current liabilities with it...,"working_capital_to_total_assets, current_assets_total_assets",NaN,Generated feature 'short_term_funding_capacity' is redundant: |rho|=0.989 ag...
9,2,income_efficiency_ratio,"This evaluates how efficiently total income is converted into net income, in...","net_income_to_total_assets, total_income_total_expense",0.0007,None


In [8]:
ran = ledger[ledger["error"].isna()]
print(ran.iloc[0]["code"] if len(ran) else "no proposal ran cleanly")

widened = discovered.dataset
print(f"\ncandidates now: {len(widened.new_features)} = {len(dataset.new_features)} declared "
      f"+ {len(discovered.kept_features)} discovered")

# (operating_expense_to_total_assets_ratio, Ratio of operating expense to total assets)
# Usefulness: This feature captures the proportion of an organization's operating expenses in relation to its total asset base, offering insights into efficiency and cost management.
# Input samples: 'df["operating_expense_rate"]': [0.0, 4320000000.0, 4530000000.0], 'df["total_asset_growth_rate"]': [4990000000.0, 6520000000.0, 7200000000.0]
df["operating_expense_to_total_assets_ratio"] = df["operating_expense_rate"] / df["total_asset_growth_rate"].where(df["total_asset_growth_rate"] != 0)

candidates now: 18 = 13 declared + 5 discovered


<a id='3'></a>
## 3. XGBoost validation

Every candidate - declared and discovered alike - goes through the same five stages, on the full splits:

| Stage | Question | Config |
| --- | --- | --- |
| 1. data quality | is the column usable, and shaped the same in valid / test as in train? | `data_quality.*` |
| 2. feature selection | does it carry signal the incumbents do not already carry? | `feature_selection.*` |
| 3. model builds | XGBoost `base`, `base_plus_new`, and `leave_one_in` - the incumbents plus one candidate | `model.*` |
| 4. analysis | Gini gain per variant on valid and test; SHAP rank | `analysis.*` |
| 5. verdict | four gates per candidate, then KEEP or TRY A NEW BATCH for the batch | `verdict.*` |

The widened dataset from section 2 is passed straight in, so discovery does not run again. From the command line, sections 2 and 3 are one run: `mllite -c configs/bankruptcy.yaml --set discovery.enabled=true`.

In [9]:
cfg_validate = load_config(CONFIG, {"run.name": "usage_validation", "run.log_level": "WARNING"})
result = Pipeline(cfg_validate).run(dataset=widened)
print(f"{result.elapsed_seconds:.0f}s -> {result.output_dir}")

37s -> outputs/usage_validation/20260915_105422


### The verdict

Four gates per candidate. `run.gates: open` measures every candidate at every gate and removes nothing, so the table shows where each one would have fallen; `enforce` makes the gates act.

| Gate | Fails when | Threshold |
| --- | --- | --- |
| `data quality` | the column itself is unusable | `data_quality.*` |
| `feature selection` | no signal, or signal the incumbents already carry | `feature_selection.*` |
| `gini gain` | its `leave_one_in` model did not beat `base` on test and valid | `verdict.min_gini_gain` |
| `shap rank` | the model kept it but barely uses it | `verdict.max_shap_rank_pct` |

In [10]:
gate_cols = ["feature", "verdict", "failed_at", "data quality", "feature selection","gini gain", "shap rank", "gini_gain_test", "reason"]
verdicts = result.verdicts[gate_cols].copy()
verdicts.insert(1, "source", ["discovered" if f in discovered.kept_features else "declared"
                              for f in verdicts["feature"]])
verdicts.round(4)

,feature,source,verdict,failed_at,data quality,feature selection,gini gain,shap rank,gini_gain_test,reason
0,cash_turnover_rate,declared,FAIL,feature selection,PASS,FAIL,FAIL,FAIL,0.0178,signal screen: weak spearman vs target (0.0172) | gini gain +0.0178 on test ...
1,cash_total_assets,declared,FAIL,gini gain,PASS,PASS,FAIL,PASS,0.0175,"gini gain +0.0175 on test (valid -0.0066), below +0.0050"
2,financial_risk_index,discovered,FAIL,gini gain,PASS,PASS,FAIL,PASS,0.0135,"gini gain +0.0135 on test (valid -0.0011), below +0.0050"
3,income_efficiency_ratio,discovered,FAIL,feature selection,PASS,FAIL,FAIL,PASS,0.0129,redundancy screen: redundant with total_income_total_expense (|rho|=0.914) |...
4,cash_flow_ratio,discovered,FAIL,gini gain,PASS,PASS,FAIL,FAIL,0.0125,"gini gain +0.0125 on test (valid +0.0007), below +0.0050 | SHAP rank 73 of 1..."
5,operating_expense_to_total_assets_ratio,discovered,FAIL,feature selection,PASS,FAIL,FAIL,FAIL,0.0114,signal screen: weak spearman vs target (-0.0033) | gini gain +0.0114 on test...
6,cash_flow_to_total_assets,declared,FAIL,feature selection,PASS,FAIL,FAIL,PASS,0.0111,redundancy screen: redundant with cash_flow_to_sales (|rho|=0.971) | gini ga...
7,cash_flow_to_equity,declared,FAIL,feature selection,PASS,FAIL,FAIL,FAIL,0.0108,redundancy screen: redundant with cash_flow_to_sales (|rho|=0.955) | gini ga...
8,liquidity_comparison_ratio,discovered,FAIL,gini gain,PASS,PASS,FAIL,FAIL,0.0107,"gini gain +0.0107 on test (valid -0.0180), below +0.0050 | SHAP rank 71 of 1..."
9,cand_noise,declared,FAIL,feature selection,PASS,FAIL,FAIL,FAIL,0.0105,signal screen: weak spearman vs target (-0.0066) | gini gain +0.0105 on test...


In [11]:
print(json.dumps(result.batch.summary(), indent=2))

{
  "verdict": "TRY A NEW BATCH",
  "n_candidates": 18,
  "n_passed": 0,
  "passed": [],
  "failed_at": {
    "feature selection": 10,
    "gini gain": 8
  },
  "batch_gini_gain": 0.00589,
  "note": "No candidate cleared all four gates - most fell at the feature selection. Propose a different batch rather than relaxing the thresholds. Gates are OPEN (run.gates: open): this is advisory only - nothing was removed, and every candidate was measured at all four gates."
}


In [12]:
cols = ["variant", "n_features", "adj_gini_valid", "adj_gini_test", "gini_gain_valid", "gini_gain_test"]
result.analysis.comparison[cols].sort_values("gini_gain_test", ascending=False).round(4).head(12)

,variant,n_features,adj_gini_valid,adj_gini_test,gini_gain_valid,gini_gain_test
14,loi__cash_turnover_rate,85,0.9072,0.9033,-0.0154,0.0178
13,loi__cash_total_assets,85,0.9160,0.9030,-0.0066,0.0175
16,loi__financial_risk_index,85,0.9215,0.8990,-0.0011,0.0135
17,loi__income_efficiency_ratio,85,0.9150,0.8983,-0.0076,0.0129
7,loi__cash_flow_ratio,85,0.9233,0.8980,0.0007,0.0125
19,loi__operating_expense_to_total_assets_ratio,85,0.9159,0.8969,-0.0067,0.0114
11,loi__cash_flow_to_total_assets,85,0.9036,0.8966,-0.0190,0.0111
8,loi__cash_flow_to_equity,85,0.9180,0.8962,-0.0046,0.0108
18,loi__liquidity_comparison_ratio,85,0.9046,0.8962,-0.0180,0.0107
3,loi__cand_noise,85,0.9180,0.8960,-0.0045,0.0105


In [13]:
# Where the candidates land in the base_plus_new model's SHAP ranking.
ranking = result.analysis.feature_ranking.query("variant == 'base_plus_new'")
ranking[ranking["feature"].isin(result.dataset.new_features)][
    ["feature", "mean_abs_shap", "shap_share", "shap_rank"]].round(4)

,feature,mean_abs_shap,shap_share,shap_rank
97,cash_total_assets,0.0075,0.0200,14
110,cash_current_liability,0.0029,0.0078,27
111,cash_flow_per_share,0.0027,0.0072,28
115,income_efficiency_ratio,0.0024,0.0064,32
124,cash_flow_to_total_assets,0.0015,0.0040,41
127,financial_risk_index,0.0011,0.0030,44
129,cash_flow_rate,0.0010,0.0027,46
139,cash_reinvestment_pct,0.0007,0.0018,56
140,operating_expense_to_total_assets_ratio,0.0007,0.0018,57
142,cash_flow_to_equity,0.0006,0.0016,59


#### One feature at a time: SHAP rank in its base + one model

The table above ranks the candidates inside `base_plus_new`, where they compete with each other as well as with the incumbents - a candidate can take credit that would otherwise go to a correlated one. Each candidate also has its own `leave_one_in` model, `loi__<feature>`: the incumbents plus that one feature. Its rank there is the cleanest read of how much the model uses it when nothing else new is present.

`rank_pct` is the rank as a fraction of the model's features. `ratio` is its SHAP share against an equal split - 1.0 means it pulls exactly its headcount weight. Each row comes from a different model, so `mean_abs_shap` is on that model's own scale: compare rows by `rank_pct` or `ratio`, not by raw SHAP. `gini_gain_test` is the same model's gain over `base`, shown alongside because attribution is not value. The verdict's SHAP gate reads `verdict.shap_variant` (`base_plus_new` here); this is the per-feature view.

In [14]:
comparison = result.analysis.comparison.set_index("variant")
rows = []
for feature in result.dataset.new_features:
    variant = f"loi__{feature}"
    shap = result.analysis.shap_ranking.query("variant == @variant")
    if feature not in set(shap["feature"]):
        continue          # no base + one model for it
    row = shap.set_index("feature").loc[feature]
    rows.append({
        "feature": feature,
        "source": "discovered" if feature in discovered.kept_features else "declared",
        "shap_rank": int(row["shap_rank"]),
        "of": len(shap),
        "rank_pct": row["shap_rank"] / len(shap),
        "mean_abs_shap": row["mean_abs_shap"],
        "shap_share": row["shap_share"],
        "ratio": row["shap_share"] * len(shap),
        "gini_gain_test": comparison.loc[variant, "gini_gain_test"],
    })

pd.DataFrame(rows).sort_values("shap_rank").reset_index(drop=True).round(4)

,feature,source,shap_rank,of,rank_pct,mean_abs_shap,shap_share,ratio,gini_gain_test
0,cash_flow_to_sales,declared,8,85,0.0941,0.1894,0.0292,2.4839,0.0043
1,cash_flow_per_share,declared,9,85,0.1059,0.1572,0.0298,2.5306,0.0084
2,cash_flow_rate,declared,12,85,0.1412,0.1476,0.0222,1.8872,0.0002
3,cash_flow_to_equity,declared,12,85,0.1412,0.1273,0.0260,2.2088,0.0108
4,cash_flow_to_liability,declared,12,85,0.1412,0.1583,0.0249,2.1183,0.0070
5,cfo_to_assets,declared,14,85,0.1647,0.1230,0.0228,1.9361,0.0102
6,cash_flow_to_total_assets,declared,16,85,0.1882,0.0157,0.0165,1.4033,0.0111
7,cash_total_assets,declared,18,85,0.2118,0.0810,0.0201,1.7056,0.0175
8,cash_reinvestment_pct,declared,23,85,0.2706,0.0587,0.0112,0.9559,0.0091
9,cash_current_liability,declared,25,85,0.2941,0.0550,0.0106,0.8968,0.0105


In [15]:
# Everything the run wrote; report.md is the human-readable summary.
for path in sorted(result.output_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(result.output_dir))

batch_verdict.json
candidate_verdicts.csv
config.resolved.yaml
data_quality_report.csv
feature_ranking.csv
feature_selection_decisions.json
feature_selection_mi_matrix.csv
feature_selection_redundancy.csv
feature_selection_spearman_matrix.csv
feature_selection_target_stats.csv
feature_selection_verdicts.csv
metrics_by_variant_split.csv
models/base.json
models/base_plus_new.json
models/loi__cand_dup_roa_c.json
models/loi__cand_noise.json
models/loi__cash_current_liability.json
models/loi__cash_flow_per_share.json
models/loi__cash_flow_rate.json
models/loi__cash_flow_ratio.json
models/loi__cash_flow_to_equity.json
models/loi__cash_flow_to_liability.json
models/loi__cash_flow_to_sales.json
models/loi__cash_flow_to_total_assets.json
models/loi__cash_reinvestment_pct.json
models/loi__cash_total_assets.json
models/loi__cash_turnover_rate.json
models/loi__cfo_to_assets.json
models/loi__financial_risk_index.json
models/loi__income_efficiency_ratio.json
models/loi__liquidity_comparison_ratio.js

<a id='4'></a>
## 4. The loop

One pass answers one question - did this batch earn a place? The loop turns the answer into the next batch:

- **KEEP**: the candidates that cleared all four gates join the incumbent set. The next batch has to beat a stronger model, and the proposer can build on what was kept.
- **TRY A NEW BATCH**: nothing joins. The proposer is told what was tried and where it failed, so it proposes something different instead of the same ideas again.

Discovery remembers every proposal *within* a run, across its `max_rounds`; *across* iterations, that memory travels in the task description. Each iteration is one `Pipeline.run` with discovery on and nothing declared by hand, and the kept columns - already computed on every split - are carried forward in the frames. It stops after `MAX_ITERATIONS`, or after `PATIENCE` iterations in a row that kept nothing.

In [ ]:
MAX_ITERATIONS, PATIENCE = 3, 2

incumbents = list(dataset.base_features)       # grows with every KEEP
frames = dict(dataset.frames)                  # carries each kept column forward
briefing = cfg.discovery.task_description
tried, history, misses = [], [], 0

for iteration in range(1, MAX_ITERATIONS + 1):
    cfg_loop = load_config(CONFIG, {"run.name": f"usage_loop_{iteration}",
                                    "run.log_level": "WARNING", "discovery.enabled": True})
    cfg_loop.features.base = list(incumbents)
    cfg_loop.features.new = []                 # only what the proposer brings
    if tried:
        cfg_loop.discovery.task_description = (
            briefing + "\n\nAlready tried in earlier iterations - propose something "
            "different: " + "; ".join(tried))

    run = Pipeline(cfg_loop).run(frames=frames)
    kept = list(run.batch.passed)
    for _, row in run.verdicts.iterrows():
        history.append({"iteration": iteration, "feature": row["feature"],
                        "verdict": row["verdict"], "failed_at": row["failed_at"],
                        "gini_gain_test": row["gini_gain_test"]})
        outcome = "kept" if row["feature"] in kept else f"failed at {row['failed_at']}"
        tried.append(f"{row['feature']} ({outcome})")

    print(f"iteration {iteration}: {run.batch.verdict} - {len(kept)} of "
          f"{run.batch.n_candidates} kept{': ' + ', '.join(kept) if kept else ''}")
    if kept:
        incumbents += kept
        frames = dict(run.dataset.frames)      # the kept columns, on every split
        misses = 0
    else:
        misses += 1
        if misses >= PATIENCE:
            print(f"stopping: {PATIENCE} iteration(s) in a row kept nothing")
            break

pd.DataFrame(history).round(4)

In [ ]:
added = incumbents[len(dataset.base_features):]
print(f"incumbent set: {len(dataset.base_features)} -> {len(incumbents)} features")
print("added:", added or "nothing")

Each iteration writes a full run directory under `outputs/usage_loop_<n>/`: `discovered_features.csv` is that iteration's proposal ledger, and `report.md` its verdict.

**Before acting on a KEEP.** The thresholds live in the config (`verdict.min_gini_gain`, `verdict.require_valid_too`, `verdict.max_shap_rank_pct`), but they are blunt: with ~44 events in test a 0.005 gain sits inside the noise band, so repeat across seeds before trusting one. And nothing here detects leakage - a feature built from the outcome clears every gate and posts a large gain.

**Your own data.** Prepare the materials with `data/cdss_us_sbs/prepare.ipynb`, fill in its config, and point `CONFIG` at it.